# Run Stateful Agents with Runloop Devboxes

This cookbook shows how to use the OpenAI Agents SDK with a Runloop devbox for a stateful, multi-turn workload. The example stages tax-document PDFs over time, suspends the devbox while idle, wakes and resumes the same filesystem for follow-up documents, verifies the final filing package with a second agent, and snapshots the finished workspace for auditability.

The full runnable CLI is included in [`runloop_w2filer_devbox_lifecycle.py`](runloop_w2filer_devbox_lifecycle.py). The notebook explains the flow and calls the script so the long orchestration code stays reusable outside Jupyter.

## What you will build

The host process creates a Runloop devbox and mounts a scoped workspace with deterministic helper scripts and sample PDFs. A `SandboxAgent` uses the shell capability inside the devbox to install missing PDF tooling, parse staged documents, and refresh a Form 1040-style JSON package. Between intake turns, the host serializes session state, closes the session with `pause_on_exit=True`, then resumes the same devbox after the next document is selected.

The lifecycle is:

1. Create a Runloop devbox with a manifest rooted at `/root/project`.
2. Upload the initial W-2 PDF into `taxpayer_data/documents/`.
3. Run the tax-prep agent to produce `output/tax_return_summary.json` and `output/f1040_form_data.json`.
4. Suspend the devbox while waiting for more tax documents.
5. Wake and resume the same devbox, upload another PDF, and refresh the filing in place.
6. Hand the persisted workspace to a filing-review agent, write `output/filing_checklist.md`, and take a Runloop disk snapshot.

## Prerequisites

Install this example's dependencies in your local environment:

```bash
cd openai-cookbook/examples/agents_sdk/runloop-devbox-lifecycle
python -m pip install -r requirements.txt
```

Set your credentials in the host environment before running the live demo:

```bash
export OPENAI_API_KEY=...
export RUNLOOP_API_KEY=...
```

Keep keys on the host. The script mounts only the helper files, task configuration, staged PDFs, and output directories that the sandbox needs.

In [ ]:
from pathlib import Path

example_dir = Path.cwd()
if not (example_dir / "runloop_w2filer_devbox_lifecycle.py").exists():
    example_dir = (Path.cwd() / "examples" / "agents_sdk" / "runloop-devbox-lifecycle").resolve()

script_path = example_dir / "runloop_w2filer_devbox_lifecycle.py"
data_dir = example_dir / "data"

print(script_path)
print(sorted(path.name for path in data_dir.iterdir()))

## Run the lifecycle demo

The command below runs non-interactively with the bundled W-2 as the initial document and the bundled 1099 as an additional staged document. The script will create a Runloop devbox, suspend it between intake turns, resume it for the second document, verify the final artifacts, snapshot the disk, and clean up the devbox at the end.

In [ ]:
import os
import subprocess
import sys

required_env = ["OPENAI_API_KEY", "RUNLOOP_API_KEY"]
missing = [name for name in required_env if not os.environ.get(name)]

if missing:
    print(f"Skipping live run. Set {', '.join(missing)} in the host environment first.")
else:
    subprocess.run(
        [
            sys.executable,
            str(script_path),
            "--additional-w2-path",
            str(data_dir / "sample_1099.pdf"),
        ],
        check=True,
    )

## Useful CLI options

Run the script directly when you want repeatable terminal output or want to integrate this pattern into another harness:

```bash
python runloop_w2filer_devbox_lifecycle.py \
  --model gpt-5.4 \
  --filing-status single \
  --qualifying-children 0 \
  --w2-path data/sample_w2.pdf \
  --additional-w2-path data/sample_1099.pdf
```

Optional flags include `--blueprint-name` for a Runloop blueprint override and `--download-dir` to copy sandbox `output/` artifacts locally after the final review prompt.

## Adapting the pattern

The tax helpers make this demo deterministic, but the lifecycle pattern is general: keep orchestration and credentials in the host, mount only task-scoped files into the devbox, suspend idle stateful work, wake on demand, validate outputs from the host, and snapshot important checkpoints. The same shape works for databases, dev servers, long-running pipelines, document processing, and code agents that need a durable execution environment.